# 🎬 rzdhop AI — Studio backend on Kaggle / Colab

This notebook runs the **FastAPI backend**, which also serves the **Studio
dashboard**, behind an ngrok tunnel.

**How it works:**
1. Add your secrets (section 2), then run all cells
2. Open the **Public URL** that appears — the dashboard is served there
3. Sign in with the **API token** the server prints when it starts
4. Start clipping!

If `npm` is not available on the machine, the dashboard is not built; the API
still works, and its interactive docs are at `<Public URL>/docs`.

---

## 1. Setup Project

In [ ]:
# Clone repository
!rm -rf ./* ./.*
!git clone https://github.com/rzdhop/opensource-clipping-better.git .

In [ ]:
%%capture
# Install dependencies
!pip install -r requirements.txt
!pip install pyngrok nest-asyncio aiofiles

In [ ]:
# Setup system dependencies
!apt-get -qq update && apt-get -qq install -y ffmpeg

# Build the dashboard so the backend can serve it at the public URL.
import shutil
if shutil.which("npm"):
    !cd web/dashboard && npm ci --no-audit --no-fund && npm run build
else:
    print("⚠️ npm not found: the dashboard is not built. The API still works at <Public URL>/docs")

## 2. Configure API Keys

Make sure you have added these secrets in **Add-ons > Secrets**:
- `GROQ_API_KEY` and/or `GOOGLE_API_KEY` — at least one; both are free
  ([Groq](https://console.groq.com/keys), [Gemini](https://aistudio.google.com/apikey))
- `NGROK_AUTHTOKEN` — ngrok auth token (get from https://dashboard.ngrok.com)
- `NVIDIA_API_KEY` — (optional) a backup link in the provider chain
- `API_TOKEN` — (optional) pins the dashboard token; otherwise a new one is printed each run
- `PEXELS_API_KEY` — (optional) for B-roll footage
- `HF_TOKEN` — (optional) for speaker diarization

In [ ]:
import os
from pathlib import Path

# Try Kaggle secrets first, then Colab
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    def get_secret(name, default=""):
        try:
            return secrets.get_secret(name) or default
        except:
            return default
    print("✅ Using Kaggle Secrets")
except ImportError:
    try:
        from google.colab import userdata
        def get_secret(name, default=""):
            try:
                return userdata.get(name) or default
            except:
                return default
        print("✅ Using Colab Secrets")
    except ImportError:
        def get_secret(name, default=""):
            return os.environ.get(name, default)
        print("⚠️ Using environment variables")

# Load secrets
GROQ_API_KEY = get_secret("GROQ_API_KEY")
GOOGLE_API_KEY = get_secret("GOOGLE_API_KEY")
NVIDIA_API_KEY = get_secret("NVIDIA_API_KEY")
NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN")
API_TOKEN = get_secret("API_TOKEN")
PEXELS_API_KEY = get_secret("PEXELS_API_KEY")
HF_TOKEN = get_secret("HF_TOKEN")

# Set environment variables (only the ones that have a value)
for _name, _value in {
    "GROQ_API_KEY": GROQ_API_KEY, "GOOGLE_API_KEY": GOOGLE_API_KEY,
    "NVIDIA_API_KEY": NVIDIA_API_KEY, "API_TOKEN": API_TOKEN,
    "PEXELS_API_KEY": PEXELS_API_KEY, "HF_TOKEN": HF_TOKEN,
}.items():
    if _value:
        os.environ[_name] = _value

# Create .env file
env_text = f"""# Auto-generated from notebook secrets
GROQ_API_KEY={GROQ_API_KEY}
GOOGLE_API_KEY={GOOGLE_API_KEY}
NVIDIA_API_KEY={NVIDIA_API_KEY}
PEXELS_API_KEY={PEXELS_API_KEY}
HF_TOKEN={HF_TOKEN}
"""
Path(".env").write_text(env_text, encoding="utf-8")

# Status
print(f"  GROQ_API_KEY: {'✅ Set' if GROQ_API_KEY else '⚪ Not set'}")
print(f"  GOOGLE_API_KEY: {'✅ Set' if GOOGLE_API_KEY else '⚪ Not set'}")
print(f"  NVIDIA_API_KEY: {'✅ Set' if NVIDIA_API_KEY else '⚪ Not set (optional backup)'}")
print(f"  NGROK_AUTHTOKEN: {'✅ Set' if NGROK_AUTHTOKEN else '❌ Missing'}")
print(f"  PEXELS_API_KEY: {'✅ Set' if PEXELS_API_KEY else '⚪ Not set (optional)'}")
print(f"  HF_TOKEN: {'✅ Set' if HF_TOKEN else '⚪ Not set (optional)'}")

# The analysis runs on a chain of free providers. It needs Groq or Gemini:
# a job whose only key is NVIDIA's is refused before it starts.
if not (GROQ_API_KEY or GOOGLE_API_KEY):
    print("\n⚠️  Neither GROQ_API_KEY nor GOOGLE_API_KEY is set -- the analysis will refuse to start.")
    print("   Groq:   https://console.groq.com/keys")
    print("   Gemini: https://aistudio.google.com/apikey")

## 3. Start Backend Server + Tunnel

This cell starts the FastAPI server and creates an ngrok tunnel.

**Open the Public URL** in your browser and sign in with the API token the
server prints as it starts (`🔑 API token: ...`).

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

# Start ngrok tunnel first
ngrok.set_auth_token(NGROK_AUTHTOKEN)
public_url = ngrok.connect(8000)

print()
print("╔" + "═" * 58 + "╗")
print("║" + " 🌐 BACKEND SERVER READY!".ljust(58) + "║")
print("║" + "".ljust(58) + "║")
print("║" + f"  Public URL: {public_url.public_url}".ljust(58) + "║")
print("║" + "".ljust(58) + "║")
print("║" + "  📋 Open the URL above in your browser and sign".ljust(58) + "║")
print("║" + "     in with the API token printed below.".ljust(58) + "║")
print("╚" + "═" * 58 + "╝")
print()
print("⚠️  The server is running below. DO NOT STOP THIS CELL!")

# Import the FastAPI app
from web.api.app import app

# Start FastAPI server in foreground (blocking) inside Jupyter
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)
await server.serve()